# Data Quality & Validation

In [6]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))

In [7]:
# testing the connection
with engine.connect() as connection:
    print("PostgreSQL connection successful!")

PostgreSQL connection successful!


In [29]:
import pandas as pd

# Import validation engine
%run data_validation.ipynb


def run_analytics_pipeline():

    # 1. Load customer-level analytical dataset
    customer_df = pd.read_sql(
        "SELECT * FROM customer_360",
        engine
    )

    # 2. Validate incoming data
    clean_df, error_df = validate_dataframe(customer_df)

    # 3. Save invalid records for investigation
    if not error_df.empty:

        error_df.to_csv(
            "Outputs/corrupt_records_log.csv",
            index=False
        )

        print(
            f"⚠️ Flagged {len(error_df)} invalid records. "
            f"Saved to Outputs/corrupt_records_log.csv"
        )

    # 4. Continue analysis using validated records only
    run_statistical_tests(clean_df)

    generate_health_scores(clean_df)

In [9]:
from scipy.stats import chi2_contingency

table = [
    [8, 151],    # Business
    [1, 13],     # Enterprise
    [33, 440],   # Growth
    [44, 510]    # Starter
]

chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)
print("Expectd frequency:", expected)

Chi-square statistic: 1.6161202835044577
p-value: 0.6557402024616013
Degrees of freedom: 3
Expectd frequency: [[ 11.395      147.605     ]
 [  1.00333333  12.99666667]
 [ 33.89833333 439.10166667]
 [ 39.70333333 514.29666667]]


Churn vs Industry

In [10]:
from scipy.stats import chi2_contingency

table = [
    [7, 106],    # Education
    [11, 191],   # Financial Services
    [14, 162],   # Healthcare
    [7, 82],     # Logistics
    [10, 141],   # Manufacturing
    [14, 150],   # Retail
    [23, 282]    # Software
]

chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)
print("Expected frequency:", expected)

Chi-square statistic: 1.8833885113468554
p-value: 0.9301047054906803
Degrees of freedom: 6
Expected frequency: [[  8.09833333 104.90166667]
 [ 14.47666667 187.52333333]
 [ 12.61333333 163.38666667]
 [  6.37833333  82.62166667]
 [ 10.82166667 140.17833333]
 [ 11.75333333 152.24666667]
 [ 21.85833333 283.14166667]]


Country wise

In [11]:
from scipy.stats import chi2_contingency

table = [
    [14, 91],    # Australia
    [7, 109],    # Brazil
    [5, 132],    # Canada
    [8, 119],    # France
    [12, 112],   # Germany
    [14, 104],   # India
    [5, 119],    # Japan
    [8, 118],    # Netherlands
    [3, 101],    # United Kingdom
    [10, 109]    # United States
]

chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)
print("Expected frequency:", expected)


Chi-square statistic: 19.102458327769593
p-value: 0.024335672001983135
Degrees of freedom: 9
Expected frequency: [[  7.525       97.475     ]
 [  8.31333333 107.68666667]
 [  9.81833333 127.18166667]
 [  9.10166667 117.89833333]
 [  8.88666667 115.11333333]
 [  8.45666667 109.54333333]
 [  8.88666667 115.11333333]
 [  9.03       116.97      ]
 [  7.45333333  96.54666667]
 [  8.52833333 110.47166667]]


pvalue<0.05 = we reject null hypothesis

Which countries are contributing most to the significant Country → Churn association?


In [12]:
import numpy as np
from scipy.stats import chi2_contingency

countries = [
    "Australia", "Brazil", "Canada", "France", "Germany",
    "India", "Japan", "Netherlands", "United Kingdom", "United States"
]

table = np.array([
    [14, 91],
    [7, 109],
    [5, 132],
    [8, 119],
    [12, 112],
    [14, 104],
    [5, 119],
    [8, 118],
    [3, 101],
    [10, 109]
])

chi2, p_value, dof, expected = chi2_contingency(table)

residuals = (table - expected) / np.sqrt(expected)

for country, residual in zip(countries, residuals[:, 0]):
    print(f"{country}: {residual:.2f}")

Australia: 2.36
Brazil: -0.46
Canada: -1.54
France: -0.37
Germany: 1.04
India: 1.91
Japan: -1.30
Netherlands: -0.34
United Kingdom: -1.63
United States: 0.50


In [13]:
import pandas as pd

query = """
SELECT
    subscription_status,
    user_count
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

print(df.head())
print(df.dtypes)

  subscription_status  user_count
0              active           3
1              active          11
2              active          35
3              active         108
4              active          20
subscription_status      str
user_count             int64
dtype: object


In [14]:
from scipy.stats import mannwhitneyu

churned = df.loc[
    df["subscription_status"] == "churned",
    "user_count"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "user_count"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 45280.5
p-value: 0.39651614228189047


There is not sufficient statistical evidence that user-count distributions differ between churned and non-churned customers in this dataset.

In [15]:
query = """
SELECT
    subscription_status,
    mrr
FROM customer_360
"""
with engine.connect() as connection:
    df = pd.read_sql(query, connection)

print(df.head())
print(df.dtypes)

  subscription_status      mrr
0              active     36.0
1              active    288.0
2              active    840.0
3              active  14852.0
4              active    648.0
subscription_status        str
mrr                    float64
dtype: object


In [16]:
churned = df.loc[
    df["subscription_status"] == "churned",
    "mrr"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "mrr"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 45771.0
p-value: 0.49086143023843753


In [17]:
query = """
SELECT
    subscription_status,
    seats
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "seats"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "seats"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 45846.0
p-value: 0.5062178937671258


In [18]:
query = """
SELECT
    subscription_status,
    total_invoiced
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "total_invoiced"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "total_invoiced"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 45607.5
p-value: 0.4587942961950079


In [19]:
query = """
SELECT
    subscription_status,
    employee_count
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "employee_count"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "employee_count"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 45569.0
p-value: 0.4511845381743994


In [20]:
query = """
SELECT
    subscription_status,
    users_per_seat
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "users_per_seat"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "users_per_seat"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 44551.5
p-value: 0.27791845868908716


In [21]:
query = """
SELECT
    subscription_status,
    COALESCE(ticket_count, 0) AS ticket_count
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "ticket_count"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "ticket_count"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 44293.0
p-value: 0.2390577411850029


In [22]:
query = """
SELECT
    subscription_status,
    COALESCE(high_priority_tickets, 0) AS high_priority_tickets
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "high_priority_tickets"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "high_priority_tickets"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 47736.0
p-value: 0.9545764106800472


In [23]:
query = """
SELECT
    subscription_status,
    COALESCE(urgent_tickets, 0) AS urgent_tickets
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "urgent_tickets"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "urgent_tickets"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 46641.0
p-value: 0.6043824178213881


In [24]:
query = """
SELECT
    subscription_status,
    avg_satisfaction
FROM customer_360
WHERE avg_satisfaction IS NOT NULL
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "avg_satisfaction"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "avg_satisfaction"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 50374.0
p-value: 0.14784654655538257


In [25]:
query = """
SELECT
    subscription_status,
    avg_resolution_hours
FROM customer_360
WHERE avg_resolution_hours IS NOT NULL
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "avg_resolution_hours"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "avg_resolution_hours"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 42206.5
p-value: 0.20382565534870756


In [26]:
query = """
SELECT
    subscription_status,
    COALESCE(unresolved_tickets, 0) AS unresolved_tickets
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "unresolved_tickets"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "unresolved_tickets"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 48495.0
p-value: 0.8162084129100544


In [27]:
import numpy as np
from scipy.stats import chi2_contingency

table = np.array([
    [14, 91],    # Australia
    [7, 109],    # Brazil
    [5, 132],    # Canada
    [8, 119],    # France
    [12, 112],   # Germany
    [14, 104],   # India
    [5, 119],    # Japan
    [8, 118],    # Netherlands
    [3, 101],    # United Kingdom
    [10, 109]    # United States
])

chi2, p_value, dof, expected = chi2_contingency(table)

n = table.sum()
r, k = table.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r - 1, k - 1))
)

print("Cramer's V:", cramers_v)

Cramer's V: 0.12616939118954854
